# 🥇 Gold Layer - Inventory & Production Analytics
## AWS S3 External Storage via Unity Catalog

**Purpose**: Create business-ready inventory and production analytics

**Source**: `workspace.`workspace-adventureworks-silver`` (S3)
**Target**: `workspace.`workspace-adventureworks-gold`` (S3)

**Gold Tables to Create**:
1. **gold_inventory_turnover** - Inventory turnover metrics by product
2. **gold_production_efficiency** - Work order efficiency metrics
3. **gold_product_performance** - Product performance scorecard

**Business Metrics**:
- Inventory turnover rate
- Work order completion rate
- Scrap rate and quality metrics
- Product profitability analysis

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import datetime
import time

# Configuration
SILVER_SCHEMA = "workspace.`workspace-adventureworks-silver`"
GOLD_SCHEMA = "workspace.`workspace-adventureworks-gold`"

print("=" * 80)
print("🥇 GOLD LAYER - INVENTORY & PRODUCTION ANALYTICS")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print(f"Source: {SILVER_SCHEMA} (AWS S3)")
print(f"Target: {GOLD_SCHEMA} (AWS S3)")
print("=" * 80)
print()

transformation_results = []

In [0]:
# Inventory Turnover Analytics
print("\n" + "=" * 80)
print("📦 BUILDING gold_inventory_turnover")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_sales_detail = spark.table(f"{SILVER_SCHEMA}.fact_sales_order_detail")
    fact_work_order = spark.table(f"{SILVER_SCHEMA}.fact_work_order")
    dim_product = spark.table(f"{SILVER_SCHEMA}.dim_product").filter(col("is_current") == True)
    dim_date = spark.table(f"{SILVER_SCHEMA}.dim_date")
    
    # Calculate sales volume by product and year
    sales_volume = (fact_sales_detail
        .join(dim_date, fact_sales_detail.order_date_sk == dim_date.date_sk)
        .groupBy(col("product_sk"), col("year"))
        .agg(
            sum("order_quantity").alias("total_units_sold"),
            sum("line_total_amount").alias("total_sales_revenue")
        )
    )
    
    # Calculate production volume by product and year
    production_volume = (fact_work_order
        .join(dim_date, fact_work_order.start_date_sk == dim_date.date_sk)
        .groupBy(col("product_sk"), col("year"))
        .agg(
            sum("order_quantity").alias("total_units_produced"),
            sum("stocked_quantity").alias("total_units_stocked"),
            sum("scrapped_quantity").alias("total_units_scrapped")
        )
    )
    
    # Combine and calculate inventory metrics
    gold_inventory_turnover = (sales_volume
        .join(production_volume, ["product_sk", "year"], "full_outer")
        .join(dim_product, sales_volume.product_sk == dim_product.dim_product_sk)
        .select(
            col("product_id"),
            col("product_name"),
            col("product_number"),
            col("category_name"),
            col("standard_cost"),
            col("list_price"),
            col("year"),
            coalesce(col("total_units_sold"), lit(0)).alias("units_sold"),
            coalesce(col("total_units_produced"), lit(0)).alias("units_produced"),
            coalesce(col("total_units_stocked"), lit(0)).alias("units_stocked"),
            coalesce(col("total_units_scrapped"), lit(0)).alias("units_scrapped"),
            coalesce(col("total_sales_revenue"), lit(0)).alias("sales_revenue")
        )
        .withColumn("inventory_turnover_rate", 
            when(col("units_stocked") > 0, col("units_sold") / col("units_stocked")).otherwise(None)
        )
        .withColumn("scrap_rate_pct",
            when(col("units_produced") > 0, col("units_scrapped") / col("units_produced") * 100).otherwise(0)
        )
        .withColumn("production_efficiency_pct",
            when(col("units_produced") > 0, col("units_stocked") / col("units_produced") * 100).otherwise(0)
        )
        .withColumn("avg_revenue_per_unit",
            when(col("units_sold") > 0, col("sales_revenue") / col("units_sold")).otherwise(None)
        )
        .withColumn("profit_margin_per_unit",
            col("avg_revenue_per_unit") - col("standard_cost")
        )
    )
    
    # Write to Gold
    (gold_inventory_turnover
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.gold_inventory_turnover")
    )
    
    duration = time.time() - start_time
    row_count = gold_inventory_turnover.count()
    
    transformation_results.append({
        "table": "gold_inventory_turnover",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_inventory_turnover created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({"table": "gold_inventory_turnover", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Production Efficiency Analytics
print("\n" + "=" * 80)
print("🏭 BUILDING gold_production_efficiency")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_work_order = spark.table(f"{SILVER_SCHEMA}.fact_work_order")
    dim_product = spark.table(f"{SILVER_SCHEMA}.dim_product").filter(col("is_current") == True)
    dim_date = spark.table(f"{SILVER_SCHEMA}.dim_date")
    
    # Calculate production metrics
    gold_production_efficiency = (fact_work_order
        .join(dim_product, fact_work_order.product_sk == dim_product.dim_product_sk)
        .join(
            dim_date.select(col("date_sk").alias("start_date_sk"), col("year")),
            fact_work_order.start_date_sk == col("start_date_sk")
        )
        .groupBy(
            col("product_id"),
            col("product_name"),
            col("category_name"),
            col("year")
        )
        .agg(
            count("work_order_id").alias("total_work_orders"),
            sum("order_quantity").alias("total_units_ordered"),
            sum("stocked_quantity").alias("total_units_completed"),
            sum("scrapped_quantity").alias("total_units_scrapped"),
            avg("order_quantity").alias("avg_order_quantity"),
            avg("stocked_quantity").alias("avg_stocked_quantity")
        )
        .withColumn("completion_rate_pct",
            when(col("total_units_ordered") > 0, 
                col("total_units_completed") / col("total_units_ordered") * 100
            ).otherwise(0)
        )
        .withColumn("scrap_rate_pct",
            when(col("total_units_ordered") > 0,
                col("total_units_scrapped") / col("total_units_ordered") * 100
            ).otherwise(0)
        )
        .withColumn("quality_score",
            when(col("total_units_completed") > 0,
                (col("total_units_completed") - col("total_units_scrapped")) / col("total_units_completed") * 100
            ).otherwise(0)
        )
    )
    
    # Write to Gold
    (gold_production_efficiency
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.gold_production_efficiency")
    )
    
    duration = time.time() - start_time
    row_count = gold_production_efficiency.count()
    
    transformation_results.append({
        "table": "gold_production_efficiency",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_production_efficiency created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({"table": "gold_production_efficiency", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Product Performance Scorecard
print("\n" + "=" * 80)
print("🎖️ BUILDING gold_product_performance")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_sales_detail = spark.table(f"{SILVER_SCHEMA}.fact_sales_order_detail")
    dim_product = spark.table(f"{SILVER_SCHEMA}.dim_product").filter(col("is_current") == True)
    dim_date = spark.table(f"{SILVER_SCHEMA}.dim_date")
    
    # Calculate comprehensive product metrics
    gold_product_performance = (fact_sales_detail
        .join(dim_product, fact_sales_detail.product_sk == dim_product.dim_product_sk)
        .join(dim_date, fact_sales_detail.order_date_sk == dim_date.date_sk)
        .groupBy(
            col("product_id"),
            col("product_name"),
            col("product_number"),
            col("category_name"),
            col("color"),
            col("size"),
            col("weight"),
            col("standard_cost"),
            col("list_price"),
            col("year")
        )
        .agg(
            count("sales_order_id").alias("order_count"),
            sum("order_quantity").alias("total_units_sold"),
            sum("line_total_amount").alias("total_revenue"),
            avg("unit_price").alias("avg_selling_price"),
            min("unit_price").alias("min_selling_price"),
            max("unit_price").alias("max_selling_price"),
            stddev("unit_price").alias("price_std_dev"),
            avg("unit_price_discount").alias("avg_discount_rate"),
            countDistinct("customer_sk").alias("unique_customers")
        )
        .withColumn("revenue_rank",
            dense_rank().over(Window.partitionBy("year", "category_name").orderBy(col("total_revenue").desc()))
        )
        .withColumn("volume_rank",
            dense_rank().over(Window.partitionBy("year", "category_name").orderBy(col("total_units_sold").desc()))
        )
        .withColumn("profit_margin_pct",
            when(col("avg_selling_price") > 0,
                (col("avg_selling_price") - col("standard_cost")) / col("avg_selling_price") * 100
            ).otherwise(0)
        )
        .withColumn("price_variance_pct",
            when(col("avg_selling_price") > 0,
                col("price_std_dev") / col("avg_selling_price") * 100
            ).otherwise(0)
        )
        .withColumn("performance_score",
            (col("revenue_rank") + col("volume_rank")) / 2
        )
    )
    
    # Write to Gold
    (gold_product_performance
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.gold_product_performance")
    )
    
    duration = time.time() - start_time
    row_count = gold_product_performance.count()
    
    transformation_results.append({
        "table": "gold_product_performance",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_product_performance created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({"table": "gold_product_performance", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Summary
print("\n" + "=" * 80)
print("📊 INVENTORY & PRODUCTION ANALYTICS SUMMARY")
print("=" * 80)

success_count = sum(1 for r in transformation_results if r["status"] == "success")
failed_count = sum(1 for r in transformation_results if r["status"] == "failed")
total_rows = sum(r["row_count"] for r in transformation_results)

print(f"\nGold Tables Created: {len(transformation_results)}")
print(f"  ✅ Success: {success_count}")
print(f"  ❌ Failed: {failed_count}")
print(f"Total Rows Created: {total_rows:,}")

summary_df = spark.createDataFrame([Row(**r) for r in transformation_results])
print("\nDetailed Results:")
display(summary_df.orderBy("status", "table"))

print("\n" + "=" * 80)
if failed_count == 0:
    print("✅ GOLD INVENTORY ANALYTICS COMPLETED SUCCESSFULLY")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit("SUCCESS")
else:
    print("⚠️ GOLD INVENTORY ANALYTICS COMPLETED WITH ERRORS")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit(f"PARTIAL_SUCCESS: {failed_count} tables failed")